In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
from default_risk.scripts.auxiliar_eda_function import check_invariant
import default_risk.config as cfg
import logging
import dtale
import dtale.global_state as dtale_global
import gc

dtale_global.cleanup()
gc.collect()

log = logging.getLogger('werkzeug')

column_order_reference="MONTHS_BALANCE"

cash_balance_df = pd.read_csv(cfg.POS_CASH_BALANCE)

cash_balance_df.sort_values(["SK_ID_PREV",column_order_reference],inplace=True)

data_frame_size= len(cash_balance_df)

#aux_function

def get_full_sorted_serie_rows(rows : pd.DataFrame):
   return recreate_and_sort_series_given_rows(rows,cash_balance_df, "SK_ID_PREV",column_order_reference)

def get_full_sorted_serie_ids(ids : list):
   return recreate_and_sort_the_serie_given_ids(ids,cash_balance_df, "SK_ID_PREV" ,column_order_reference)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)



Invariants:
Note: the # with a number is the celd of this notebook where the relevant code is for that rasoning.

1- If the status is complete (NAME_CONTRACT_STATUS == "Completed"), then CNT_INSTALMENT and CNT_INSTALMENT_FUTURE have a defined value
((CNT_INSTALMENT != null) & (CNT_INSTALMENT_FUTURE != null)) (100%) #3

2 - If the status is active (NAME_CONTRACT_STATUS == "Active"), the missing values of CNT_INSTALMENT and CNT_INSTALMENT_FUTURE are perfectly aligned
if ((CNT_INSTALMENT == null) then (CNT_INSTALMENT_FUTURE == null)) & if (CNT_INSTALMENT_FUTURE == null) then (CNT_INSTALMENT == null) (100%) #3

Soft constraints:

1- If the status is active (NAME_CONTRACT_STATUS == "Active"), then you have CNT_INSTALMENT defined (and for invariant #2 we know that also means we have CNT_INSTALMENT_FUTURE defined) (99.99%).
Anomalies: We found two strong patterns:
    A- where it is the first row of the sequence, so the status is active but behaves like a "Signed"
    B- data corruption. #7




Decisions summary:
Note: when we refer to order in the sequence, it is sorted by "MONTH_BALANCE".

Data Quality & Imputation:

1- If the status is active (NAME_CONTRACT_STATUS == "Active") and CNT_INSTALMENT | CNT_INSTALMENT_FUTURE is missing, we will proceed in one of these 3 ways:
    a- if this pattern appears only once and corresponds to the first row in the sequence, we will impute the NAME_CONTRACT_STATUS as "Signed"
    b- if this pattern shows up more than one time, we will flag that temporal series as "corrupted"
    (any other pattern is out of the train dataset, so we can't take an informed decision about it.) #6

2- In the row with status "Completed", the field "CNT_INSTALLMENT" has a different meaning, and represents "how long the loan lasted", and based on invariant #1 we consider it reliable and will use it as a source of truth to capture the total duration of the loan. #5

3- We will consider the original expected duration of the loan to be the first value of CNT_INSTALMENT that is not null in the temporal series. #4

4- We will consider instalments paid in advance when we register a decreasing jump in CNT_INSTALMENT_FUTURE a decrease larger than one between consecutive observations of that loan. #4

5- If the series never has a row marked as completed, we will flag it as "not_closed", with these subcases:
    a- if CNT_INSTALMENT_FUTURE also never reaches 0, but the last register with "MONTH_BALANCE" is >= -3, then we will consider it as "ongoing"
    b- if CNT_INSTALMENT_FUTURE also never reaches 0, and the last register has a value of "MONTH_BALANCE" < -3, then we will consider it as an "incomplete series" #4

6- If CNT_INSTALMENT_FUTURE decreases until it reaches 0, all the rows with status active and CNT_INSTALMENT_FUTURE = 0 that are between the last status with CNT_INSTALMENT_FUTURE != 0 and the first row marked as "Completed" will be considered a delay-tail if they do not have changes in SK_DPD or SK_DPD_DEF. We will capture it as a metric but will not consider it part of the duration of the original loan. #9

7- We will consider dead-tail to be all the rows with status "Completed" after the first one with status "Completed". #9

8- To detect staggered counters and corrupted data, we expect a minimum amount of different states in CNT_INSTALMENT_FUTURE in a "completed" loan.
So we will explain the definition with the following example.
For instance: a loan originally planned for 10 months with 10 rows needs a minimum of 10 different states.
We can also extend this to edge cases.
If the loan was planned for 10 months (decision #3) but is paid in advance (check decision #4) and has 4 rows, we expect at least 4 different values in CNT_INSTALMENT_FUTURE.
And if the loan was originally planned for 10 months (decision #3) and lasts more than that (rescheduling of the debt), we expect at least 10 different values in that field. #10

9- Conversely, we expect to observe changes in CNT_INSTALMENT only for rescheduling or instalments paid in advance. Therefore, too many changes suggest either a problematic loan with various reschedules or data corruption.
So we will capture the amount of unique values in CNT_INSTALMENT and define an initial heuristic threshold for data corruption of 5 (in this dataset, all cases above that amount of unique values correspond to corrupted data). #11

## Invariants:
1- If the status is complete (NAME_CONTRACT_STATUS == "Completed"), then CNT_INSTALMENT and CNT_INSTALMENT_FUTURE have a defined value ((CNT_INSTALMENT != NaN) & (CNT_INSTALMENT_FUTURE != NaN)) (100%) #3

2 - If the status is active (NAME_CONTRACT_STATUS == "Active"), the missing values of CNT_INSTALMENT and CNT_INSTALMENT_FUTURE are perfectly aligned if ((CNT_INSTALMENT == NaN) then (CNT_INSTALMENT_FUTURE == NaN)) & if (CNT_INSTALMENT_FUTURE == NaN) then (CNT_INSTALMENT == NaN) (100%) #3

## Soft constraints:
1- If the status is active (NAME_CONTRACT_STATUS == "Active"), then CNT_INSTALMENT is typically a non NaN value.  (and for invariant #2 we know that also means we have CNT_INSTALMENT_FUTURE defined)
(99.99%).
Anomalies: We found two strong patterns:
    A- Cases that occur only in the first row of the sequence, where the status is "Active" but the behavior is identical to "Signed" 
    B- instances of clear data corruption. #7

## Decisions summary:
Note: when we refer to order in a sequence, it is sorted by "MONTH_BALANCE".

#### I. Data Quality & Imputation

1- If the status is active (NAME_CONTRACT_STATUS == "Active") and CNT_INSTALMENT | CNT_INSTALMENT_FUTURE is missing, we will proceed using these heuristics:
    a- if this pattern occurs only in the first row of the sequence, we will impute the NAME_CONTRACT_STATUS as "Signed"
    b- if this pattern appears more than one time, we will flag the time series as "corrupted"
    (any other pattern fall outside of the training dataset) #6

2- To detect staggered counters and corrupted data, we expect a minimum amount of unique values in CNT_INSTALMENT_FUTURE in a "completed" loan.
So we will explain the definition with the following example.

For instance: a loan originally planned for 10 months with 10 rows needs a minimum of 10 different states.
We can also extend this to edge cases.
If the loan was planned for 10 months (decision #3) but is paid in advance (check decision #4) and has 4 rows, we expect at least 4 different values in CNT_INSTALMENT_FUTURE.
And if the loan was originally planned for 10 months (decision #3) and lasts more than that (rescheduling of the debt), we expect at least 10 different values in that field. #10

3- Conversely, we expect to observe changes in CNT_INSTALMENT only for rescheduling or instalments paid in advance. Therefore, too many changes suggest either a problematic loan with various reschedules or data corruption.
So we will capture the amount of unique values in CNT_INSTALMENT and define an initial heuristic threshold for data corruption of 5 (in this dataset, all cases above that amount of unique values correspond to corrupted data). #11

#### II. Loan Duration Dynamics

1- In the row with status "Completed", the field "CNT_INSTALMENT" has a different meaning, and represents "how long the loan actually lasted", and based on invariant #1 we consider it reliable and will use it as a source of truth to capture the total duration of the loan. #5

2- We define the original expected duration of the loan as the first non-null value of CNT_INSTALMENT within the temporal series. #4

3- If the series never reaches the status of "Completed", we will flag it as "not_closed", under these subcases:
    a- if CNT_INSTALMENT_FUTURE also never reaches 0, but the most recent "MONTH_BALANCE" is >= -3, then we will consider it as "ongoing"
    b- if CNT_INSTALMENT_FUTURE also never reaches 0, and most recent register is "MONTH_BALANCE" < -3, then we will consider it as an "incomplete series" #4

#### III. Anomalies & Payment Behavior

1- We will consider instalments paid in advance when we register a decreasing jump in CNT_INSTALMENT_FUTURE a decrease larger than one between consecutive observations of that loan. #4

2- If CNT_INSTALMENT_FUTURE decreases until it reaches 0, all the rows with status active and CNT_INSTALMENT_FUTURE = 0 that are between the last status with CNT_INSTALMENT_FUTURE != 0 and the first row marked as "Completed" will be considered a delay-tail if they do not have changes in SK_DPD or SK_DPD_DEF. We will capture it as a metric but will not consider it part of the duration of the original loan. #9

3- We define as dead-tail as all the rows with status "Completed" after the first one with status "Completed". #9

In [ ]:
#1
#files for the data dictionary
create_files_nulls_per_colmun(cash_balance_df,"POS_CASH_balance")

In [ ]:
#2
#run the screening script on POS_CASH_balance
eda_per_table_printing_results(cash_balance_df, schema, "POS_CASH_balance",False)

In [ ]:
#3
check_invariant(((cash_balance_df["NAME_CONTRACT_STATUS"] == "Completed") & (cash_balance_df["CNT_INSTALMENT"].isnull())),"the status is Completed and the CNT_INSTALMENT is null",data_frame_size)

check_invariant(((cash_balance_df["NAME_CONTRACT_STATUS"] == "Completed") & (cash_balance_df["CNT_INSTALMENT_FUTURE"].isnull())),"the status is Completed and the CNT_INSTALMENT_FUTURE is null",data_frame_size)

check_invariant(((cash_balance_df["CNT_INSTALMENT_FUTURE"].isnull()) & (cash_balance_df["CNT_INSTALMENT"].notna()) & (cash_balance_df["NAME_CONTRACT_STATUS"] == "Active")), "the status is active, CNT_INSTALMENT_FUTURE is null and CNT_INSTALMENT is not null",data_frame_size)

check_invariant(((cash_balance_df["CNT_INSTALMENT"].isnull()) & (cash_balance_df["CNT_INSTALMENT_FUTURE"].notna()) & (cash_balance_df["NAME_CONTRACT_STATUS"] == "Active")),"the status is active, CNT_INSTALMENT is null and  CNT_INSTALMENT_FUTURE is not null",data_frame_size)

check_invariant((cash_balance_df["CNT_INSTALMENT"].isnull()) &  (cash_balance_df["NAME_CONTRACT_STATUS"] == "Active"), "the CNT_INSTALMENT is null and the status is active" ,data_frame_size)

In [ ]:

previous_contracts_id_with_nulls=cash_balance_df[cash_balance_df["CNT_INSTALMENT"].isnull()]["SK_ID_PREV"]
rows_of_contracts_with_null=cash_balance_df[cash_balance_df["SK_ID_PREV"].isin(previous_contracts_id_with_nulls)]
len(rows_of_contracts_with_null)

In [ ]:
#4
#how the series looks like without seleccion biases (limitating to 1000 first to handle better the visualization)
first_ids_prev_app= (cash_balance_df["SK_ID_PREV"].unique())[:1000]
slice_of_applications=get_full_sorted_serie_ids(first_ids_prev_app)
dtale.show(slice_of_applications)


In [ ]:
#5
#in order to understand the nature of the missing values in this table we will visualize the series with nulls
previous_contracts_rows_with_nulls=cash_balance_df[cash_balance_df["CNT_INSTALMENT"].isnull()]
sorted_list=get_full_sorted_serie_rows(previous_contracts_rows_with_nulls)
dtale.show(sorted_list)
#this show the correlation between missing in CNT_INSTALMENT and the status of the contract. CNT_INSTALLMENT seems to be the expected amount total of installment. Sometimes we can see
#how in the row of the secuence that mark the loan as "completed" (NAME_CONTRACT_STATUS == "Completed") the field change to the actual duration of the loan. Meaning the client made payments in advance
#and also we can see the increment pointing to a reschedule of the total duration of the loan. 


In [ ]:
#6
#when we craft the data dictionary we discover that the amount of missing values in CNT_INSTALMENT and  CNT_INSTALMENT_FUTURE have a very small diference, suggesting
#for semantic and that values, that their missing values are highly correlated but not perfectly aligned. So now we proceed to analize 
#where the missing values of CNT_INSTALMENT and CNT_INSTALMENT_FUTURE are not aligned 
not_aligned_nulls_future_defined= ((cash_balance_df["CNT_INSTALMENT"].isnull()) & (cash_balance_df["CNT_INSTALMENT_FUTURE"].notna()))
not_aligned_nulls_counter_defined = ((cash_balance_df["CNT_INSTALMENT_FUTURE"].isnull()) & (cash_balance_df["CNT_INSTALMENT"].notna()) )
rows_to_analize= cash_balance_df[(not_aligned_nulls_future_defined | not_aligned_nulls_counter_defined)]
print(len(rows_to_analize))
get_full_sorted_serie_rows(rows_to_analize).head(50)
#the pattern show that, this tends to happend at the first part of the serie, where the status is "Signed" o diferent from "active". Where this info maybe don't exist at the same time.
#we also will check if that happend with "active" status to define it as invariant.


In [ ]:
#7
#now lets visualize the entire series that have at least one row with status "Active - Complete" and CNT_INSTALMENT in null.
states_with_no_nulls=["Active","Completed"]
inconsistency_status_mask=cash_balance_df["CNT_INSTALMENT"].isnull()  &  cash_balance_df["NAME_CONTRACT_STATUS"].isin(states_with_no_nulls)
#this type of nulls (in active contracts) are candidates to be filled 

rows_with_null=cash_balance_df[inconsistency_status_mask]
dtale.show(get_full_sorted_serie_rows(rows_with_null))
#from here we can deduce a fill rule for this non exepecteable nulls. 1- if the sequence have an row with active and "CNT_INSTALMENT" == NULL, often are the first one, a likely registration error easy
#to correct. Putting the status as "Signed" at that point following the pattern that the rest applications folow in this table. But if have more than one row of the secuence with "CNT_INSTALMENT" == NULL can
#assume is prove of data corruption or a canceled contract, someting we want to ignore in the agg metrics.

In [ ]:
#8
#Based on the last visualizations, we detect a very intersting case that could be useful as heuristic to detect series with anormal behavior. When the "CNT_INSTALMENT_FUTURE" is 0 more than one time, because
#this only should happend at the moment the contract is marked as "Completed"
ceros_df = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: (g["CNT_INSTALMENT_FUTURE"] == 0).sum() > 1)
pd.set_option('display.max_columns', None)
dtale.show(ceros_df)
#this help us to detect the "double countability" of the final month. Sometimes they put "Active" with "CNT_INSTALMENT_FUTURE" == 0 and just after another row marking the loan as "Completed"
#and have no day past due. That's mean they are countabilizating twice the final row with the row "active" and "completed" with "CNT_INSTALMENT_FUTURE" == 0. Now we want to analize excluding those cases
#that are easy to correct.

In [ ]:
#9
#Series with "CNT_INSTALMENT_FUTURE" == 0 in more than one row, without counting the "Completed" row, because is the "excpectable" 0 of the serie. This avoid the "Double countability" just detected
#in the last vizualization.
no_completed_status_ceros_df = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: ((g["CNT_INSTALMENT_FUTURE"] == 0)&(g["NAME_CONTRACT_STATUS"] != "Completed")).sum() > 1)
no_completed_status_ceros_df.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
pd.set_option('display.max_columns', None)
dtale.show(no_completed_status_ceros_df)
#this visualization is very useful to recognize two things 1- the "dead tails" of some loans, where  te "CNT_INSTALMENT_FUTURE" is 0, there is any change in any relevant variable but the loan have more rows after that,
#so we can't see any activity but the loan still having more observations, with the status of "active" or "completed".  We will chatch this with a feature but discard it for the agg metrics. 2-Some corrupted data
#of CNT_INSTALMENT_FUTURE being 0 across all the temporal serie. We will desing the features to differences this two type of cases.

In [ ]:
#10
counter_of_unique_values = cash_balance_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_FUTURE"].nunique()
ids_application_constant_value= counter_of_unique_values[counter_of_unique_values == 1].index
id_mask = cash_balance_df["SK_ID_PREV"].isin(ids_application_constant_value)
series_with_constant_value= cash_balance_df[id_mask].copy()
series_with_constant_value.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"], inplace=True)
del id_mask, counter_of_unique_values
gc.collect()
dtale.show(series_with_constant_value)
#Excluding records with a single row (which typically represent recent applications), we can observe a numerous instances of stagnant counters. This suggests to be corrupted data based on CNT_INSTALMENT_FUTURE
#expectable behavior (monotonically decreasing).
#This can be generalizated further as a rule if CNT_INSTALMENT_FUTURE must to contain a minumun ammout of unique values determinated by the length of the series.
#For instance: A loan originally planned for 10 months with 10 rows, needs a minimum of 10 different states. 
#Also we can extend this for edge cases.
#If the loan was planned for 12 months and have 4 rows, need at least 4 unique values (paid in advance).
#and if the Loan was originally planned for 10 months and last more than that (rescheduling of the debt) we expect at least 10 different values in that field. 

In [ ]:
searched_status= ["Approved","Singed"]
status_mask=cash_balance_df["NAME_CONTRACT_STATUS"].isin(searched_status)
id_contracts_with_status=cash_balance_df[status_mask]["SK_ID_PREV"].unique()

print(len(id_contracts_with_status))

print(len(cash_balance_df["SK_ID_PREV"].unique()))  


In [ ]:
len(ceros_df["SK_ID_PREV"].unique())


In [ ]:
#11
#Another think that seems very useful is the ammount of changes in "CNT_INSTALMENT". How is the "expected" total amount of installment at that point of the loan, means a replanification of the loan
#o inconsistency (there is series where this number change constanly, a clear sign of data corruption)

ceros_in_expected_instalment = cash_balance_df.groupby("SK_ID_PREV").filter(lambda g: (len(g["CNT_INSTALMENT"].unique()))> 5)
dtale.show(ceros_in_expected_instalment)
